<a href="https://colab.research.google.com/github/oliviagcadwell/ann-cancer-classification-deeplearning/blob/main/Neural_network_deep_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd # data manipulation
import numpy as np # number manipulation
import matplotlib.pyplot as plt # plotting
import seaborn as sns # plotting
from sklearn.metrics import classification_report, confusion_matrix # evaluation

In [ ]:
# Wisconsin Diagnostic Breast Cancer dataset (569 tumors, 30 features), bundled with scikit-learn
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)

In [ ]:
# scikit-learn codes the target as 0 = malignant, 1 = benign.
# Flip it so 1 = malignant, the class we want to detect.
df = data.frame.rename(columns={'target': 'malignant'})
df['malignant'] = 1 - df['malignant']
df.head()

In [ ]:
# goal: predict whether a tumor is malignant (1) or benign (0)
# using the other 30 measurements as predictors

In [ ]:
df.describe().transpose()

In [ ]:
df.info()

In [ ]:
sns.countplot(x='malignant', hue='malignant', data=df, palette='viridis', legend=False)
plt.show()

In [ ]:
# correlation analysis, how is it correlated with dependent variable
df.corr()['malignant'].sort_values().plot(kind = "bar")

In [ ]:
# correlation frame
sns.heatmap(df.corr())

In [ ]:
x=df.drop('malignant', axis=1)
# x matrix all the data. except our dependent variable

y=df['malignant'].values

# ys is the dependent variable

In [ ]:
x.shape
# dimension
# 569 rows and 30 columns

In [ ]:
y.shape

In [ ]:
y

In [ ]:
# for model buildling we need to split data into training set and test set
from sklearn.model_selection import train_test_split

In [ ]:
x_train_full, x_test, y_train_full, y_test = train_test_split(x, y,
                                                    test_size=0.25,
                                                    random_state=101)

# hold out 20% of the training data as a validation set for early stopping,
# so the test set is only used once, for the final evaluation
x_train, x_val, y_train, y_val = train_test_split(x_train_full, y_train_full,
                                                  test_size=0.2,
                                                  random_state=101)

In [ ]:
# Preprocessing - rescales each feature to the 0-1 range so no feature dominates because of its units
from sklearn.preprocessing import MinMaxScaler

In [ ]:
scaler = MinMaxScaler()
# defining the scalar object

In [ ]:
# fit the scaler on the training data only, then apply it to the other sets
x_train = scaler.fit_transform(x_train)

In [ ]:
x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)

In [ ]:
# finally start the deep learning part - artificial neural network libraries
# ANN libraries
# packages designed for deep learning

from tensorflow.keras.models import Sequential #sequential version of ANN
from tensorflow.keras.layers import Dense, Dropout

import tensorflow as tf
tf.keras.utils.set_random_seed(42)  # makes results reproducible

In [ ]:
# building ANN
model = Sequential()
model.add(Dense(30, activation='relu')) #first neuronal layer, 30 is the number of nodes
model.add(Dense(15, activation='relu')) #second neuronal layer
model.add(Dense(1, activation='sigmoid')) #output layer
model.compile(loss='binary_crossentropy', optimizer='adam')

In [ ]:
# model fitting - an epoch is one full pass through the training data
model.fit(x=x_train, y=y_train, epochs=600,
          validation_data=(x_val, y_val))

In [ ]:
# Losses mean how the actual data of our dependent variable is different from the predicted data
# if the number is big there is a big discrepancy between what the data is saying versus what we predicted

losses = pd.DataFrame(model.history.history)
losses.plot()

In [ ]:
# training loss (blue) keeps falling: the model fits the training data better and better
# validation loss rises after a point: the model is overfitting
# next: stop training at the right time and reduce complexity

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
# stops training once validation loss stops improving, so the model doesn't overfit

In [ ]:
early_stop = EarlyStopping(monitor='val_loss',
                           mode='min', verbose=1, patience=29)

In [ ]:
# building ANN
model = Sequential()
model.add(Dense(30, activation='relu')) #first neuronal layer, 30 is the number of nodes
model.add(Dense(15, activation='relu')) #second neuronal layer
model.add(Dense(1, activation='sigmoid')) #output layer
model.compile(loss='binary_crossentropy', optimizer='adam')

In [ ]:
model.fit(x=x_train, y=y_train, epochs=600,
          validation_data=(x_val, y_val),
          callbacks = [early_stop])

In [ ]:
losses = pd.DataFrame(model.history.history)
losses.plot()

In [ ]:
# this can also be used for reducing the complexity of your model
# building ANN
model = Sequential()
model.add(Dense(30, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(15, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam')


In [ ]:
early_stop = EarlyStopping(monitor='val_loss',
                           mode='min', verbose=1, patience=25)

In [ ]:
model.fit(x=x_train, y=y_train, epochs=600,
          validation_data=(x_val, y_val),
          callbacks = [early_stop])

In [ ]:
losses = pd.DataFrame(model.history.history)
losses.plot()

In [ ]:
# with dropout + early stopping, training and validation loss stay close together

In [ ]:
# finally, predict on the held-out test set; probability > 0.5 = malignant
# outputs the probability of whether it is malignant of benign

predictions = model.predict(x_test)
class_predictions = (predictions>0.5).astype('int')

In [ ]:
class_predictions

In [ ]:
# final evaluation on the test set, which the model never saw during training
print(classification_report(y_test, class_predictions,
                            target_names=['Benign', 'Malignant']))

In [ ]:
cm = confusion_matrix(y_test, class_predictions)

In [ ]:
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
            xticklabels=["Predicted Benign", "Predicted Malignant"],
            yticklabels=["Actual Benign", "Actual Malignant"])
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.title("Confusion Matrix Cancer")
plt.show()